In [ ]:
"""
===============================================================================
Title       : DTLA Parameter Converter
Description : A Tkinter-based GUI tool to process CSV files by extracting
              DTLA Status parameters from binary representation and saving the
              transformed data into a new CSV file.
Author      : Sanmathi S
Date        : 2026
Version     : 1.0
===============================================================================
Features:
    - Browse and select input CSV file.
    - Extract first 8 bits from 'dtlastatus' column and split into 4 parameters.
    - Save processed data to an output CSV file.
    - Modern GUI with progress bar and status updates.
    - Error handling and user-friendly messages.

Dependencies:
    - Python 3.x
    - tkinter
    - pandas
===============================================================================
Usage:
    Run the script directly:
        python dtla_parameter_converter.py
===============================================================================
"""

import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import pandas as pd
import os
import time

# --- Data Processing Logic (Kept the same) ---

def extract_parameters(value):
    """Function to extract the first 8 bits and split into 4 parameters."""
    try:
        value = int(value)
        binary_str = format(value, '016b')
        first_8_bits = binary_str[:8]
        params = [first_8_bits[i:i+2] for i in range(0, 8, 2)]
        return [int(p, 2) for p in params]
    except (ValueError, TypeError):
        return [pd.NA] * 4

def process_file_logic(input_file, output_file, progress_var, status_var):
    """Handles the main data processing using pandas, with progress updates."""
    status_var.set("Processing: Reading file...")
    progress_var.set(10)
    try:
        df = pd.read_csv(input_file)
        status_var.set("Processing: Applying data transformations...")
        progress_var.set(40)
        
        columns = ["DTLA solenoid", "MOR switch status (DTLA)", "SOR switch status (DTLA)", "Traction switch"]
        df[columns] = df['dtlastatus'].apply(lambda x: pd.Series(extract_parameters(x)))
        
        status_var.set("Processing: Saving output file...")
        progress_var.set(80)

        df.to_csv(output_file, index=False)
        progress_var.set(100)
        status_var.set(f"Processing complete! Output saved to {output_file}")
        
        # Show success message box
        messagebox.showinfo("Success", f"Processing complete!\nOutput saved to:\n{output_file}")

    except Exception as e:
        status_var.set(f"Error: {e}")
        messagebox.showerror("Error", f"An error occurred: {e}")
    finally:
        # Reset progress bar at the end (brief delay for visual effect)
        app.after(1000, lambda: progress_var.set(0)) 

# --- Tkinter GUI Application ---

class Application(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("CSV Data Processor")
        self.geometry("680x400")
        
        # --- Modern Light Theme Configuration (Clean White/Gray) ---
        self.main_bg = '#f9f9f9'
        self.configure(bg=self.main_bg) 
        self.style = ttk.Style()
        self.style.theme_use('clam') 
        
        # Configure overall style
        self.style.configure('.', background=self.main_bg, foreground='#333333', font=("Segoe UI", 10))
        
        # Style for Buttons (Primary Color: Professional Blue)
        self.style.configure('TButton', 
                             background='#0078d4', 
                             foreground='#FFFFFF', 
                             font=("Segoe UI", 10, 'bold'), 
                             relief='flat', 
                             padding=8)
        self.style.map('TButton', background=[('active', '#005fa3')])
        
        # Style for Entry widgets (clean look with border)
        self.style.configure('TEntry', fieldbackground='#FFFFFF', foreground='#333333', borderwidth=1, relief='solid', padding=5)
        
        # Configure Progressbar with a vibrant green color
        self.style.configure("Custom.Horizontal.TProgressbar", 
                             troughcolor='#e0e0e0', 
                             background='#28a745', # Success Green
                             bordercolor=self.main_bg, 
                             lightcolor='#28a745', 
                             darkcolor='#28a745')

        self.input_file_path = tk.StringVar()
        self.output_file_path = tk.StringVar()
        self.status_var = tk.StringVar(value="Ready: Select input file to begin.")
        self.progress_var = tk.DoubleVar(value=0)

        self.create_widgets()

    def create_widgets(self):
        # Header Label with a strong blue color
        tk.Label(self, text="DTLA Parameter Converter", bg=self.main_bg, fg='#005fa3', 
                 font=("Segoe UI", 20, 'bold')).pack(pady=(30, 15))
        
        # Separator for visual hierarchy
        ttk.Separator(self, orient='horizontal').pack(fill='x', padx=20, pady=5)

        # --- Input File Section ---
        frame_input = tk.Frame(self, bg=self.main_bg)
        frame_input.pack(padx=20, pady=(15, 5), fill='x')
        tk.Label(frame_input, text="📂 Input CSV File:", bg=self.main_bg, fg='#333333', font=("Segoe UI", 12, 'bold')).pack(anchor='w', pady=5)
        
        ttk.Entry(frame_input, textvariable=self.input_file_path, width=50, style='TEntry').pack(side=tk.LEFT, fill='x', expand=True, ipady=3)
        ttk.Button(frame_input, text="Browse", command=self.browse_input).pack(side=tk.RIGHT, padx=5)

        # --- Output File Section ---
        frame_output = tk.Frame(self, bg=self.main_bg)
        frame_output.pack(padx=20, pady=10, fill='x')
        tk.Label(frame_output, text="💾 Output CSV File:", bg=self.main_bg, fg='#333333', font=("Segoe UI", 12, 'bold')).pack(anchor='w', pady=5)
        
        self.entry_output = ttk.Entry(frame_output, textvariable=self.output_file_path, width=50, style='TEntry')
        self.entry_output.pack(side=tk.LEFT, fill='x', expand=True, ipady=3)
        
        # Professional button text for selecting save location/name
        ttk.Button(frame_output, text="Save Output File", command=self.browse_output).pack(side=tk.RIGHT, padx=5)

        # --- Process Button (Using a success green color) ---
        self.style.configure('Process.TButton', 
                             background='#28a745', 
                             foreground='#ffffff',
                             font=("Segoe UI", 12, 'bold'))
        self.style.map('Process.TButton', background=[('active', '#218838')])

        ttk.Button(self, text="▶ Start Processing", command=self.run_processing, style='Process.TButton').pack(pady=30, ipadx=30, ipady=10)

        # --- Status Label ---
        tk.Label(self, textvariable=self.status_var, bg=self.main_bg, fg='#005fa3', font=("Segoe UI", 10, 'italic', 'bold')).pack(pady=5)

        # --- Progress Bar ---
        self.progress = ttk.Progressbar(self, variable=self.progress_var, maximum=100, style="Custom.Horizontal.TProgressbar", mode='determinate')
        self.progress.pack(padx=20, fill='x', pady=10)


    def browse_input(self):
        """Open a file dialog to select the input file and set the default output path."""
        filename = filedialog.askopenfilename(
            title="Select Input CSV File",
            filetypes=(("CSV files", "*.csv"), ("All files", "*.*"))
        )
        if filename:
            self.input_file_path.set(filename)
            # Set default output path to the same directory with "_output" appended
            directory, base_name = os.path.split(filename)
            name, ext = os.path.splitext(base_name)
            output_filename = f"{name}_output{ext}"
            default_output_path = os.path.join(directory, output_filename)
            self.output_file_path.set(default_output_path)
            
            self.status_var.set(f"Selected input: {os.path.basename(filename)}. Output path set by default.")
    
    def browse_output(self):
        """Allows the user to explicitly select the output file path and name."""
        initial_file = self.output_file_path.get() if self.output_file_path.get() else 'Bookd_output.csv'

        filename = filedialog.asksaveasfilename(
            title="Save Output CSV File As",
            initialfile=os.path.basename(initial_file),
            defaultextension=".csv",
            filetypes=(("CSV files", "*.csv"), ("All files", "*.*"))
        )
        if filename:
            self.output_file_path.set(filename)
            self.status_var.set(f"Output path updated: {os.path.basename(filename)}")


    def run_processing(self):
        """Initiate the processing function."""
        input_path = self.input_file_path.get()
        output_path = self.output_file_path.get()

        if not input_path:
            messagebox.showwarning("Warning", "Please select an input file first.")
            return
        
        if not output_path:
             messagebox.showwarning("Warning", "Please define an output file path.")
             return

        # Use .after() method to ensure GUI updates immediately before blocking with the pandas process.
        self.after(100, process_file_logic, input_path, output_path, self.progress_var, self.status_var)


if __name__ == "__main__":
    app = Application()
    app.mainloop()
